# 🤖 MBTI Personality Detection — RAG + LLM-based Agent

---

**Architecture: Agent = LLM + Planning + Tool Use + Loop + Memory**

| Component | Implementation |
|-----------|---------------|
| **LLM** | Qwen2.5-7B-Instruct (4-bit quantization) |
| **Planning** | LLM autonomously decides which tool to invoke, with what parameters, and when to stop |
| **Tool Use** | **Function Calling (Tool Binding)** — LLM generates structured `<tool_call>` with name + arguments; tools accept LLM-specified parameters (e.g. targeted knowledge queries) |
| **Loop** | ReAct + Function Calling: LLM reasons → generates `<tool_call>` → system executes → observation fed back → LLM decides next step |
| **Memory** | Short-term scratchpad (per-sample trace) + Long-term experience store (FAISS-indexed high-confidence classifications + reasoning traces → dynamic few-shot prompting) |

**RAG (Retrieval-Augmented Generation):**
- **Index 1 — Similar Posts Store**: Training posts embedded with `all-MiniLM-L6-v2` → FAISS → few-shot examples
- **Index 2 — MBTI Knowledge Base**: Dimension descriptions + 16-type cognitive profiles → FAISS → domain context

**Key Autonomy Feature:** The LLM controls not only *which* tool to call, but also *how* — e.g., `retrieve_knowledge(query="thinking vs feeling analytical language patterns")` generates a dynamic embedding for targeted knowledge retrieval.

**Datasets supported:**
- MBTI-500 (~106K rows): `/kaggle/input/.../MBTI 500.csv`
- Kaggle MBTI (~8600 rows): `/kaggle/input/.../mbti_1.csv`

## 📦 Cell 1 — Install Dependencies

In [ ]:
!pip install -q transformers faiss-cpu scikit-learn torch torchvision torchaudio
!pip install -q sentence-transformers bitsandbytes accelerate tqdm

## 📥 Cell 2 — Imports

In [ ]:
import os, re, json, time, warnings, gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import pipeline as hf_pipeline

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    classification_report
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
print('✅ Imports done')

## ⚙️ Cell 3 — Config

In [ ]:
# ───────────────────────────────────────────────
# CONFIG — all paths and hyperparameters here
# ───────────────────────────────────────────────
CONFIG = {
    # Paths
    'DATA_PATH_500':  '/kaggle/input/datasets/zeyadkhalid/mbti-personality-types-500-dataset/MBTI 500.csv',
    'DATA_PATH_8K':   '/kaggle/input/datasets/datasnaek/mbti-type/mbti_1.csv',
    'RESULT_DIR':     '/kaggle/working/results',

    # Data preprocessing
    'MAX_POSTS':      50,
    'MAX_WORDS':      70,

    # RAG
    'RAG_TOP_K':      5,
    'EMBED_MODEL':    'all-MiniLM-L6-v2',
    'EMBED_DIM':      384,

    # Agent
    'LLM_MODEL':      'Qwen/Qwen2.5-7B-Instruct',
    'CONFIDENCE_THRESHOLD': 0.55,
    'MAX_RETRIES':    1,

    # Long-term Memory
    'MEMORY_CONF_THRESHOLD': 0.80,   # min per-axis confidence to store experience
    'MAX_REACT_STEPS':      5,      # max LLM turns in ReAct loop

    # Batched Inference
    'BATCH_SIZE':            4,      # samples processed in parallel via batched LLM generation
}

os.makedirs(CONFIG['RESULT_DIR'], exist_ok=True)

# ── GPU detection ──
NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda' if NUM_GPUS > 0 else 'cpu')
print(f'🖥  GPUs available: {NUM_GPUS}  |  Device: {DEVICE}')

# ── Constants ──
MBTI_TYPES  = ['infj','infp','intj','intp','isfj','isfp','istj','istp',
               'enfj','enfp','entj','entp','esfj','esfp','estj','estp']
MBTI_EXTRA  = ['introvert','extrovert','introverted','extroverted',
               'sensing','intuition','thinking','feeling','judging','perceiving']
ALL_MASK    = MBTI_TYPES + MBTI_EXTRA
MASK_RE     = re.compile(r'\b(' + '|'.join(ALL_MASK) + r')\b', re.IGNORECASE)

LABEL_COL   = 'type'
TEXT_COL    = 'posts'
LABEL_COLS  = ['label_ie', 'label_sn', 'label_tf', 'label_jp']
DIM_NAMES   = ['I/E', 'S/N', 'T/F', 'J/P']
print('✅ Config ready')

## 📂 Cell 4 — Load & Preprocess Dataset

In [ ]:
# ── Load dataset (prefer 500K, fall back to 8K) ──
def load_dataset(cfg):
    for path in [cfg['DATA_PATH_500'], cfg['DATA_PATH_8K']]:
        try:
            df = pd.read_csv(path)
            print(f'✅ Loaded: {path}  shape={df.shape}')
            return df
        except FileNotFoundError:
            continue
    raise FileNotFoundError('❌ No dataset found. Add data and check paths.')

df_raw = load_dataset(CONFIG)


# ── Step 1: Map 16-class MBTI → 4 binary labels ──
def map_mbti_to_binary(mbti_str):
    m = mbti_str.strip().upper()
    return int(m[0]=='I'), int(m[1]=='N'), int(m[2]=='F'), int(m[3]=='J')

df_raw[['label_ie','label_sn','label_tf','label_jp']] = (
    df_raw[LABEL_COL].apply(lambda x: pd.Series(map_mbti_to_binary(x)))
)


# ── Step 2: Psycholinguistic masking + truncation ──
def preprocess_user_posts(raw_str, max_posts=CONFIG['MAX_POSTS'], max_words=CONFIG['MAX_WORDS']):
    """Split posts, mask MBTI keywords, truncate to max_words each."""
    posts = raw_str.split('|||')
    result = []
    for p in posts[:max_posts]:
        p = MASK_RE.sub('<type>', p.strip())
        p = ' '.join(p.split()[:max_words])
        if p:
            result.append(p)
    return result

print('⏳ Preprocessing posts (masking + truncation)...')
df_raw['processed_posts'] = df_raw[TEXT_COL].apply(preprocess_user_posts)

# Concatenated version for RAG text input
df_raw['concat_posts'] = df_raw['processed_posts'].apply(lambda x: ' ||| '.join(x))
print(f'✅ Preprocessing done. Rows: {len(df_raw)}')


# ── Step 3: Stratified split across ALL 4 axes jointly ──
df_raw['combo_label'] = df_raw[LABEL_COLS].astype(str).agg(''.join, axis=1)

train_df, temp_df = train_test_split(
    df_raw, test_size=0.4, stratify=df_raw['combo_label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['combo_label'], random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'\n📊 Split → Train:{len(train_df)} | Val:{len(val_df)} | Test:{len(test_df)}')

## 📚 Cell 5 — MBTI Knowledge Base

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  MBTI KNOWLEDGE BASE (Domain Knowledge for RAG)
# ═══════════════════════════════════════════════════════════════

MBTI_DIM_KNOWLEDGE = {
    "I/E": (
        "Introversion (I) indicators in text: reflective and introspective language, longer and more "
        "complex sentences, frequent use of 'I think', 'in my opinion', hedging words like 'perhaps' "
        "or 'maybe', discussion of inner thoughts and ideas, fewer social references, preference for "
        "deep topics over small talk, lower frequency of exclamation marks, references to solitude and "
        "personal space, self-referential language, analytical introspection.\n"
        "Extraversion (E) indicators in text: action-oriented and energetic language, shorter direct "
        "sentences, frequent exclamation marks, references to social activities and groups, use of "
        "'we/us/our', discussion of events and people, more casual and conversational tone, references "
        "to parties, gatherings, collaboration, social energy, group dynamics."
    ),
    "S/N": (
        "Sensing (S) indicators in text: concrete and specific language, references to tangible details "
        "and facts, step-by-step descriptions, present-focused or past-experience based writing, "
        "practical vocabulary, sensory words (see, hear, feel, touch), specific numbers and dates, "
        "discussion of real-world events, hands-on experiences, routine references.\n"
        "Intuition (N) indicators in text: abstract and theoretical language, metaphors and analogies, "
        "discussion of possibilities and 'what if' scenarios, future-oriented thinking, conceptual "
        "vocabulary, references to patterns and connections, philosophical discussions, creative and "
        "imaginative expressions, symbolic language, big-picture focus."
    ),
    "T/F": (
        "Thinking (T) indicators in text: logical and analytical language, cause-effect reasoning "
        "using words like 'because', 'therefore', 'logically', objective and impersonal tone, "
        "critique and debate style, fewer emotional expressions, systematic analysis, focus on truth "
        "and accuracy, direct disagreement, technical vocabulary, problem-solving focus.\n"
        "Feeling (F) indicators in text: emotional and empathetic language, personal values expression, "
        "use of feeling words ('feel', 'love', 'care', 'heart'), harmony-seeking communication, "
        "people-focused discussions, supportive and encouraging tone, references to personal impact, "
        "moral judgments based on values rather than logic, relationship-oriented content."
    ),
    "J/P": (
        "Judging (J) indicators in text: decisive and definitive language using 'should', 'must', "
        "'will', structured and organized writing, closure-seeking statements, references to plans, "
        "schedules, and goals, preference for order, list-making tendency, strong opinions stated as "
        "facts, completion-focused language, methodical approach.\n"
        "Perceiving (P) indicators in text: open-ended and exploratory language using 'maybe', "
        "'might', 'what about', flexible and adaptive writing, multiple options considered, "
        "spontaneous topic changes, casual attitude toward deadlines, curiosity-driven exploration, "
        "'going with the flow' references, incomplete thoughts, tangential thinking."
    ),
}

MBTI_TYPE_PROFILES = {
    "INTJ": "Ni-Te-Fi-Se. Strategic, independent, determined. Precise analytical language, long-term "
            "vision, minimal small talk, direct communication, confident assertions, systems thinking, "
            "rare emotional expression, future-planning focus.",
    "INTP": "Ti-Ne-Si-Fe. Analytical, curious, innovative. Complex sentence structures, explores "
            "multiple angles, theoretical vocabulary, questions everything, logical precision, "
            "detached tone, idea-driven, loves intellectual debate.",
    "ENTJ": "Te-Ni-Se-Fi. Decisive, ambitious, commanding. Direct and authoritative language, "
            "efficiency-focused, goal-oriented, structured arguments, leadership references, "
            "action-oriented planning, strategic communication.",
    "ENTP": "Ne-Ti-Fe-Si. Quick-witted, argumentative, inventive. Playful language, devil's advocate "
            "positions, humor and sarcasm, idea-jumping, debate-seeking, creative connections, "
            "challenges assumptions, energetic discourse.",
    "INFJ": "Ni-Fe-Ti-Se. Insightful, principled, compassionate. Metaphorical language, deep "
            "reflections, idealistic expressions, future-oriented, empathetic but private, symbolic "
            "thinking, meaningful connections, visionary language.",
    "INFP": "Fi-Ne-Si-Te. Idealistic, empathetic, creative. Poetic and personal language, "
            "value-driven expressions, emotional depth, authenticity focus, imaginative descriptions, "
            "identity exploration, artistic sensibility.",
    "ENFJ": "Fe-Ni-Se-Ti. Charismatic, inspiring, empathetic. Encouraging and motivational language, "
            "people-focused, visionary statements, warm tone, community references, natural "
            "leadership, harmony-building communication.",
    "ENFP": "Ne-Fi-Te-Si. Enthusiastic, creative, sociable. Energetic language, many exclamations, "
            "rapid topic shifts, passionate expressions, optimistic tone, brainstorming style, "
            "explores possibilities, emotionally expressive.",
    "ISTJ": "Si-Te-Fi-Ne. Practical, reliable, methodical. Structured factual language, duty "
            "references, traditional values, step-by-step descriptions, concrete details, "
            "rule-following, historical references.",
    "ISFJ": "Si-Fe-Ti-Ne. Caring, reliable, observant. Supportive language, detail-oriented "
            "descriptions, tradition-respecting, nurturing tone, practical helpfulness, "
            "loyalty-focused, modest expression.",
    "ESTJ": "Te-Si-Ne-Fi. Organized, direct, traditional. Clear directives, fact-based arguments, "
            "efficiency-focused, authoritative tone, rule-following references, practical leadership.",
    "ESFJ": "Fe-Si-Ne-Ti. Caring, sociable, traditional. Warm and inclusive language, social harmony "
            "focus, community references, supportive expressions, practical care, people-oriented.",
    "ISTP": "Ti-Se-Ni-Fe. Practical, observant, analytical. Concise factual language, action-focused, "
            "problem-solving oriented, minimal emotional expression, hands-on references, "
            "independent thinking.",
    "ISFP": "Fi-Se-Ni-Te. Gentle, sensitive, artistic. Aesthetic language, present-moment focus, "
            "personal values, sensory descriptions, quiet emotional depth, creative expression.",
    "ESTP": "Se-Ti-Fe-Ni. Bold, direct, perceptive. Action-oriented short sentences, present-focused, "
            "practical and energetic, humor, risk-taking references, competitive language.",
    "ESFP": "Se-Fi-Te-Ni. Spontaneous, energetic, playful. Vivid sensory language, social engagement, "
            "fun-focused, enthusiastic tone, living-in-the-moment expressions, entertaining style.",
}

print(f'✅ Knowledge base ready: {len(MBTI_DIM_KNOWLEDGE)} dimension descriptions + {len(MBTI_TYPE_PROFILES)} type profiles')

## 🔍 Cell 6 — Build RAG Indexes (FAISS + Sentence-Transformer)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  BUILD RAG INDEXES (FAISS + Sentence-Transformer)
# ═══════════════════════════════════════════════════════════════
print('⏳ Loading embedding model...')
embed_model = SentenceTransformer(CONFIG['EMBED_MODEL'], device='cpu')
EMBED_DIM = CONFIG['EMBED_DIM']

# ── Index 1: Training posts → similar-post retrieval (few-shot) ──
print('⏳ Encoding training posts for RAG index...')
train_texts_for_rag = train_df['concat_posts'].tolist()
train_labels_for_rag = train_df[LABEL_COL].tolist()
train_embeddings = embed_model.encode(
    train_texts_for_rag, batch_size=128, show_progress_bar=True,
    normalize_embeddings=True
).astype(np.float32)

post_index = faiss.IndexFlatIP(EMBED_DIM)
post_index.add(train_embeddings)
print(f'✅ Post RAG index: {post_index.ntotal} vectors')

# ── Index 2: Knowledge base chunks ──
knowledge_texts = []
knowledge_labels = []
for dim_name, text in MBTI_DIM_KNOWLEDGE.items():
    knowledge_texts.append(text)
    knowledge_labels.append(dim_name)
for type_name, profile in MBTI_TYPE_PROFILES.items():
    knowledge_texts.append(profile)
    knowledge_labels.append(type_name)

knowledge_embeddings = embed_model.encode(
    knowledge_texts, normalize_embeddings=True
).astype(np.float32)
knowledge_index = faiss.IndexFlatIP(knowledge_embeddings.shape[1])
knowledge_index.add(knowledge_embeddings)
print(f'✅ Knowledge RAG index: {knowledge_index.ntotal} chunks')

# ── Pre-encode test posts for batch retrieval ──
print('⏳ Encoding test posts...')
test_texts_for_rag = test_df['concat_posts'].tolist()
test_embeddings = embed_model.encode(
    test_texts_for_rag, batch_size=128, show_progress_bar=True,
    normalize_embeddings=True
).astype(np.float32)
print(f'✅ Test embeddings ready: {test_embeddings.shape}')

## 🧠 Cell 7 — Load LLM (Qwen2.5-7B-Instruct, 4-bit)

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  LOAD LLM: Qwen2.5-7B-Instruct (4-bit NF4 quantization)
# ═══════════════════════════════════════════════════════════════
LLM_MODEL_NAME = CONFIG['LLM_MODEL']

print(f'⏳ Loading {LLM_MODEL_NAME} (4-bit quantization)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME, trust_remote_code=True)
llm_model_raw = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

llm_pipe = hf_pipeline(
    'text-generation',
    model=llm_model_raw,
    tokenizer=llm_tokenizer,
    max_new_tokens=150,
    temperature=0.1,
    do_sample=True,
    return_full_text=False,
)
print(f'✅ LLM loaded: {LLM_MODEL_NAME}  |  dtype=4bit-nf4  |  device_map=auto')

## 🔧 Cell 8 — Tool Definitions (Function Calling) + Agent Class

**Function Calling / Tool Binding (Qwen2.5 native `<tool_call>` format):**
1. `analyze_text()` — Extract psycholinguistic features
2. `retrieve_similar(top_k)` — FAISS kNN search on training posts
3. `retrieve_knowledge(query, top_k)` — **LLM specifies search query** → dynamic embedding → FAISS search on MBTI KB
4. `recall_experience()` — Query long-term memory for past high-confidence classifications
5. `predict(type, ie, ie_conf, ...)` — Submit final MBTI prediction with structured arguments

**Agent Loop:** LLM autonomously generates `<tool_call>{"name": ..., "arguments": {...}}</tool_call>` → system executes → observation fed back as `role: tool` → LLM reasons and selects next tool or predicts

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  AGENT TOOLS (callable by the agent via Function Calling)
# ═══════════════════════════════════════════════════════════════

def tool_retrieve_similar_posts(query_embedding, top_k=CONFIG['RAG_TOP_K']):
    """Tool: Retrieve similar labeled posts from training set (few-shot examples via RAG)."""
    top_k = min(max(1, int(top_k)), 10)
    scores, indices = post_index.search(query_embedding.reshape(1, -1), top_k)
    results = []
    for idx, score in zip(indices[0], scores[0]):
        row = train_df.iloc[idx]
        results.append({
            'type': row[LABEL_COL],
            'text': row['concat_posts'][:300],
            'similarity': float(score),
        })
    return results


def tool_retrieve_knowledge(query_embedding, top_k=6):
    """Tool: Retrieve relevant MBTI knowledge from knowledge base via RAG."""
    top_k = min(max(1, int(top_k)), 8)
    scores, indices = knowledge_index.search(query_embedding.reshape(1, -1), top_k)
    results = []
    for idx, score in zip(indices[0], scores[0]):
        results.append({
            'label': knowledge_labels[idx],
            'content': knowledge_texts[idx],
        })
    return results


def tool_analyze_text(posts_text):
    """Tool: Extract psycholinguistic features from posts."""
    words = posts_text.lower().split()
    total = len(words)
    if total == 0:
        return {}
    sentences = max(posts_text.count('.') + posts_text.count('?') + posts_text.count('!'), 1)
    return {
        'total_words': total,
        'avg_word_len': round(np.mean([len(w) for w in words]), 2),
        'avg_sent_len': round(total / sentences, 1),
        'question_ratio': round(posts_text.count('?') / sentences, 3),
        'exclam_ratio': round(posts_text.count('!') / sentences, 3),
        'first_person_pct': round(
            sum(1 for w in words if w in {'i','me','my','myself','mine'}) / total * 100, 2),
        'we_pct': round(
            sum(1 for w in words if w in {'we','us','our','ourselves'}) / total * 100, 2),
        'emotion_pct': round(
            sum(1 for w in words if w in {
                'feel','feeling','love','hate','happy','sad','angry','beautiful',
                'wonderful','terrible','amazing','awful','care','heart','soul',
                'passion','joy','fear','hope','wish'
            }) / total * 100, 2),
        'thinking_pct': round(
            sum(1 for w in words if w in {
                'think','thought','logic','logical','reason','because','therefore',
                'analyze','analysis','system','theory','hypothesis','evidence','data','fact'
            }) / total * 100, 2),
        'abstract_pct': round(
            sum(1 for w in words if w in {
                'concept','idea','theory','possibility','imagine','philosophy',
                'meaning','purpose','pattern','connection','potential','vision',
                'insight','metaphor','symbol','abstract','infinite'
            }) / total * 100, 2),
        'url_count': posts_text.lower().count('http'),
    }


# ═══════════════════════════════════════════════════════════════
#  TOOL DEFINITIONS — JSON Schema for Qwen2.5 Function Calling
#  LLM generates <tool_call>{"name":..., "arguments":{...}}</tool_call>
# ═══════════════════════════════════════════════════════════════

TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "analyze_text",
            "description": (
                "Extract quantitative psycholinguistic features from the user's posts: "
                "word count, avg word/sentence length, question ratio, first-person "
                "pronoun %, we/us %, emotion word %, thinking word %, abstract word %. "
                "Call this first to get statistical signals about personality."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "retrieve_similar",
            "description": (
                "FAISS kNN semantic search to find the most similar labeled posts "
                "from the training set, with their ground-truth MBTI types, as "
                "few-shot reference examples."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "top_k": {
                        "type": "integer",
                        "description": "Number of similar posts to retrieve (1-10). Default: 5."
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "retrieve_knowledge",
            "description": (
                "Semantic search over the MBTI knowledge base to retrieve dimension "
                "descriptions and type cognitive profiles. Use a TARGETED query to "
                "find specific knowledge, e.g. 'introversion vs extraversion social "
                "energy language patterns' or 'INTP Ti-Ne analytical writing style'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": (
                            "Semantic search query for MBTI knowledge retrieval. "
                            "Be specific, e.g. 'thinking vs feeling analytical "
                            "emotional language' or 'judging perceiving structured "
                            "flexible writing patterns'."
                        )
                    },
                    "top_k": {
                        "type": "integer",
                        "description": "Number of knowledge chunks to retrieve (1-8). Default: 4."
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "recall_experience",
            "description": (
                "Query the agent's long-term memory for similar posts previously "
                "classified with high confidence, along with the reasoning traces "
                "used for those classifications."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "predict",
            "description": (
                "Submit the final MBTI prediction with per-dimension confidence "
                "scores. Call ONLY after gathering evidence from at least 2 other tools."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "type": {"type": "string", "description": "4-letter MBTI type, e.g. 'INTP'"},
                    "ie": {"type": "string", "enum": ["I", "E"], "description": "Introversion or Extraversion"},
                    "ie_conf": {"type": "number", "description": "I/E confidence, 0.5-1.0"},
                    "sn": {"type": "string", "enum": ["S", "N"], "description": "Sensing or Intuition"},
                    "sn_conf": {"type": "number", "description": "S/N confidence, 0.5-1.0"},
                    "tf": {"type": "string", "enum": ["T", "F"], "description": "Thinking or Feeling"},
                    "tf_conf": {"type": "number", "description": "T/F confidence, 0.5-1.0"},
                    "jp": {"type": "string", "enum": ["J", "P"], "description": "Judging or Perceiving"},
                    "jp_conf": {"type": "number", "description": "J/P confidence, 0.5-1.0"}
                },
                "required": ["type", "ie", "ie_conf", "sn", "sn_conf", "tf", "tf_conf", "jp", "jp_conf"]
            }
        }
    }
]


# ═══════════════════════════════════════════════════════════════
#  SYSTEM PROMPT — task-level instructions (tool schemas are
#  injected automatically by Qwen2.5's chat template)
# ═══════════════════════════════════════════════════════════════

AGENT_SYSTEM_PROMPT = (
    "You are an expert MBTI personality classifier agent. "
    "Analyze social media posts and classify the author's MBTI type "
    "along 4 dimensions by autonomously calling tools.\n\n"
    "Strategy (you decide the order and parameters):\n"
    "- Call analyze_text to get quantitative linguistic signals\n"
    "- Call retrieve_similar to find labeled posts with similar writing style\n"
    "- If uncertain about a dimension, call retrieve_knowledge with a TARGETED "
    "query (e.g. 'thinking vs feeling emotional analytical language patterns')\n"
    "- Optionally call recall_experience for past similar classifications\n"
    "- When you have enough evidence, call predict with your classification\n\n"
    "MBTI Dimensions:\n"
    "- I/E: Introversion (reflective, 'I think', hedging) vs "
    "Extraversion (social, 'we', exclamation marks)\n"
    "- S/N: Sensing (concrete, specific, practical) vs "
    "Intuition (abstract, theoretical, 'what if')\n"
    "- T/F: Thinking (logical, 'because/therefore') vs "
    "Feeling (emotional, 'feel/love/care')\n"
    "- J/P: Judging (structured, 'should/must/will') vs "
    "Perceiving (flexible, 'maybe/might')\n\n"
    "Rules:\n"
    "- Call at least 2 tools before predicting\n"
    "- Confidence: 0.5 = uncertain, 1.0 = very confident\n"
    "- For retrieve_knowledge, use SPECIFIC queries about the "
    "dimension(s) you're uncertain about"
)


# ═══════════════════════════════════════════════════════════════
#  MBTI CLASSIFICATION AGENT — Function Calling + ReAct Loop
#  Agent = LLM + Planning + Tool Use (Function Calling) + Loop + Memory
# ═══════════════════════════════════════════════════════════════

class MBTIClassificationAgent:
    """
    Single Agent for MBTI classification with genuine autonomous tool use.

    The LLM generates structured <tool_call> invocations (Qwen2.5 native format)
    with tool name AND arguments. The system executes the tool, returns the
    observation, and the LLM autonomously decides the next step.

    Key autonomy features:
      - LLM decides WHICH tool to call (not hardcoded sequence)
      - LLM decides HOW to call it (dynamic parameters, e.g. search queries)
      - LLM decides WHEN to predict (not forced after N steps)
      - retrieve_knowledge accepts LLM-generated query → dynamic embedding → FAISS

    Components:
      - LLM:            Qwen2.5-7B-Instruct (reasoning backbone)
      - Function Calling: <tool_call>{"name":..., "arguments":{...}}</tool_call>
      - Tools:           analyze_text, retrieve_similar, retrieve_knowledge,
                         recall_experience, predict
      - Memory:          Short-term scratchpad + Long-term FAISS experience store
    """

    def __init__(self, llm_pipeline, embed_model,
                 confidence_threshold=0.55, max_retries=1,
                 memory_conf_threshold=0.80, max_react_steps=10):
        self.llm = llm_pipeline
        self.embed_model = embed_model
        self.conf_threshold = confidence_threshold
        self.max_retries = max_retries
        self.memory_conf_threshold = memory_conf_threshold
        self.max_react_steps = max_react_steps

        # ── Short-term Memory (reset each sample) ──
        self.scratchpad = {}
        self.trace = []

        # ── Long-term Memory ──
        self.pattern_memory = {}
        self.experience_store = []
        self.experience_index = None
        self.total_retries = 0
        self.total_memories_stored = 0

    # ─────────────────────────────────────────
    #  LONG-TERM MEMORY: Query & Store
    # ─────────────────────────────────────────
    def _query_experience_memory(self, query_embedding, top_k=2):
        """Retrieve past high-confidence classifications from experience memory."""
        if self.experience_index is None or self.experience_index.ntotal == 0:
            return []
        k = min(top_k, self.experience_index.ntotal)
        scores, indices = self.experience_index.search(
            query_embedding.reshape(1, -1), k
        )
        results = []
        for idx, score in zip(indices[0], scores[0]):
            if idx >= 0 and score > 0.5:
                exp = self.experience_store[idx].copy()
                exp['memory_similarity'] = float(score)
                results.append(exp)
        return results

    def _store_experience(self, query_embedding, posts_text, result):
        """Store high-confidence result + LLM reasoning trace into long-term memory."""
        confs = [result.get(f'{d}_conf', 0) for d in ['ie', 'sn', 'tf', 'jp']]
        min_conf = min(confs)
        avg_conf = float(np.mean(confs))

        if min_conf < self.memory_conf_threshold:
            return False

        reasoning_parts = []
        for step in self.trace:
            if step.get('llm_generated') and step.get('thought'):
                reasoning_parts.append(step['thought'][:200])
        reasoning_summary = (
            ' → '.join(reasoning_parts)
            if reasoning_parts else 'Direct classification'
        )

        experience = {
            'type': result['type'],
            'posts_preview': posts_text[:300],
            'result': {
                k: result.get(k) for k in
                ['type', 'ie', 'ie_conf', 'sn', 'sn_conf',
                 'tf', 'tf_conf', 'jp', 'jp_conf']
            },
            'reasoning': reasoning_summary,
            'avg_conf': avg_conf,
        }

        self.experience_store.append(experience)
        emb = query_embedding.copy().reshape(1, -1).astype(np.float32)
        if self.experience_index is None:
            self.experience_index = faiss.IndexFlatIP(emb.shape[1])
        self.experience_index.add(emb)
        self.total_memories_stored += 1
        return True

    # ─────────────────────────────────────────
    #  AUTONOMOUS ReAct LOOP (Function Calling)
    # ─────────────────────────────────────────
    def classify(self, posts_text, query_embedding):
        """
        Autonomous classification loop with Function Calling:

        1. LLM sees posts → generates <tool_call> with name + arguments
        2. System executes tool → observation fed back as role='tool'
        3. LLM reasons → generates next <tool_call> or calls predict
        4. If prediction has low confidence → feedback → LLM re-reasons
        5. High-confidence results stored in long-term experience memory

        The LLM controls:
          - WHICH tool to call (autonomous selection)
          - HOW to call it (dynamic arguments, e.g. knowledge queries)
          - WHEN to predict (not forced)
        """
        self.scratchpad = {'observations': [], 'retries': 0}
        self.trace = []

        messages = [
            {'role': 'system', 'content': AGENT_SYSTEM_PROMPT},
            {'role': 'user', 'content': (
                'Classify this user\'s MBTI type based on their forum posts:\n\n'
                + posts_text[:2000]
            )},
        ]

        result = None
        predict_attempts = 0
        tools_called = set()

        for step_num in range(self.max_react_steps):
            # ── LLM generates response (with function calling) ──
            raw = self._call_llm_with_tools(messages)

            # ── Extract reasoning text (before <tool_call>) ──
            thought = self._extract_thought(raw)

            # ── Parse structured tool call ──
            tool_name, tool_args = self._parse_tool_call(raw)

            # Fallback only if LLM output is completely unparseable
            if not tool_name:
                if 'analyze_text' not in tools_called:
                    tool_name, tool_args = 'analyze_text', {}
                elif 'retrieve_similar' not in tools_called:
                    tool_name, tool_args = 'retrieve_similar', {}
                else:
                    tool_name, tool_args = 'predict', {}

            self.trace.append({
                'step': tool_name,
                'thought': thought or '(LLM generated tool call directly)',
                'tool_args': tool_args or {},
                'llm_generated': True,
            })

            # ── Handle predict action ──
            if tool_name == 'predict':
                if tool_args and ('type' in tool_args or 'ie' in tool_args):
                    result = self._validate(tool_args)
                else:
                    result = self._parse_json(raw)
                    if result:
                        result = self._validate(result)

                if result:
                    predict_attempts += 1
                    low_conf = [
                        d for d in ['ie', 'sn', 'tf', 'jp']
                        if result.get(f'{d}_conf', 0) < self.conf_threshold
                    ]

                    if not low_conf or predict_attempts > self.max_retries:
                        self.trace[-1]['observation'] = (
                            f'Final: {result["type"]} '
                            f'[I/E={result["ie"]}({result["ie_conf"]:.2f}), '
                            f'S/N={result["sn"]}({result["sn_conf"]:.2f}), '
                            f'T/F={result["tf"]}({result["tf_conf"]:.2f}), '
                            f'J/P={result["jp"]}({result["jp_conf"]:.2f})]'
                        )
                        break
                    else:
                        dim_names = {'ie': 'I/E', 'sn': 'S/N', 'tf': 'T/F', 'jp': 'J/P'}
                        low_str = ', '.join(
                            f'{dim_names[d]}={result[f"{d}_conf"]:.2f}'
                            for d in low_conf
                        )
                        obs = (
                            f'Prediction {result["type"]} has LOW CONFIDENCE on '
                            f'{low_str} (threshold={self.conf_threshold}). '
                            f'Call retrieve_knowledge with a targeted query about '
                            f'the uncertain dimension(s), then re-predict.'
                        )
                        self.trace[-1]['observation'] = obs
                        messages.append({'role': 'assistant', 'content': raw})
                        messages.append({'role': 'user', 'content': f'Observation: {obs}'})
                        self.scratchpad['retries'] += 1
                        self.total_retries += 1
                        continue
                else:
                    obs = (
                        'Could not parse prediction. Please call predict '
                        'with valid arguments: type, ie, ie_conf, sn, sn_conf, '
                        'tf, tf_conf, jp, jp_conf.'
                    )
                    self.trace[-1]['observation'] = obs
                    messages.append({'role': 'assistant', 'content': raw})
                    messages.append({'role': 'user', 'content': f'Observation: {obs}'})
                    continue

            # ── Execute tool with LLM-specified arguments ──
            tools_called.add(tool_name)
            observation = self._execute_tool(
                tool_name, tool_args, posts_text, query_embedding
            )

            if observation is None:
                obs_str = (
                    f'Unknown tool "{tool_name}". Available: '
                    f'analyze_text, retrieve_similar, retrieve_knowledge, '
                    f'recall_experience, predict'
                )
            else:
                obs_str = self._format_observation(tool_name, observation)

            self.trace[-1]['observation'] = obs_str

            # Feed back to conversation using tool role
            messages.append({'role': 'assistant', 'content': raw})
            messages.append({
                'role': 'tool', 'name': tool_name, 'content': obs_str
            })

        # ── Fallback if loop exhausted without valid prediction ──
        if result is None:
            result = self._fallback_predict(posts_text, query_embedding)
            self.trace.append({
                'step': 'fallback_predict',
                'thought': 'ReAct loop exhausted. Using direct prediction.',
                'tool_args': {},
                'observation': f'Fallback: {result.get("type", "INFP")}',
                'llm_generated': False,
            })

        # ── MEMORY UPDATE ──
        mbti = result.get('type', 'INFP')
        self.pattern_memory[mbti] = self.pattern_memory.get(mbti, 0) + 1
        stored = self._store_experience(query_embedding, posts_text, result)
        avg_conf = float(np.mean(
            [result.get(f'{d}_conf', 0) for d in ['ie', 'sn', 'tf', 'jp']]
        ))

        mem_obs = (
            f'pattern_memory["{mbti}"]={self.pattern_memory[mbti]}'
            + (f', ✅ experience #{self.total_memories_stored} stored '
               f'(avg_conf={avg_conf:.2f})'
               if stored else
               f', ⏭ not stored (conf < {self.memory_conf_threshold})')
        )
        self.trace.append({
            'step': 'memory_update',
            'thought': f'Update memory with "{mbti}".',
            'tool_args': {},
            'observation': mem_obs,
            'llm_generated': False,
        })

        return result

    # ─────────────────────────────────────────
    #  TOOL EXECUTION WITH DYNAMIC ARGUMENTS
    # ─────────────────────────────────────────
    def _execute_tool(self, name, args, posts_text, query_embedding):
        """Execute tool with LLM-specified arguments."""
        if name == 'analyze_text':
            return tool_analyze_text(posts_text)

        elif name == 'retrieve_similar':
            top_k = int(args.get('top_k', CONFIG['RAG_TOP_K']))
            return tool_retrieve_similar_posts(query_embedding, top_k=top_k)

        elif name == 'retrieve_knowledge':
            query = args.get('query', '')
            top_k = int(args.get('top_k', 4))
            if query:
                # LLM specified a custom search query →
                # embed it → targeted FAISS search
                q_emb = self.embed_model.encode(
                    [query], normalize_embeddings=True
                ).astype(np.float32)
                return tool_retrieve_knowledge(q_emb, top_k=top_k)
            else:
                return tool_retrieve_knowledge(query_embedding, top_k=top_k)

        elif name == 'recall_experience':
            return self._query_experience_memory(query_embedding)

        return None

    def _format_observation(self, action, result):
        if action == 'analyze_text':
            return json.dumps(result, indent=2)
        elif action == 'retrieve_similar':
            lines = [
                f"  {i+1}. [{s['type']}] (sim={s['similarity']:.3f}): "
                f"{s['text'][:150]}"
                for i, s in enumerate(result)
            ]
            return f"Top {len(result)} similar posts:\n" + '\n'.join(lines)
        elif action == 'retrieve_knowledge':
            lines = [
                f"  [{k['label']}]: {k['content'][:250]}"
                for k in result
            ]
            return f"{len(result)} knowledge chunks:\n" + '\n'.join(lines)
        elif action == 'recall_experience':
            if not result:
                return (f"No relevant past experiences "
                        f"(memory_size={len(self.experience_store)})")
            lines = [
                f"  [{e['type']}] (sim={e['memory_similarity']:.2f}, "
                f"conf={e['avg_conf']:.2f}): {e['reasoning'][:200]}"
                for e in result
            ]
            return f"{len(result)} past experiences:\n" + '\n'.join(lines)
        return str(result)[:500]

    # ─────────────────────────────────────────
    #  LLM INTERACTION — Function Calling
    # ─────────────────────────────────────────
    def _call_llm_with_tools(self, messages, max_new_tokens=300):
        """Call LLM with tool definitions injected via chat template."""
        try:
            text = self.llm.tokenizer.apply_chat_template(
                messages, tools=TOOL_DEFINITIONS,
                tokenize=False, add_generation_prompt=True
            )
        except TypeError:
            # Fallback: convert tool-role messages to user-role
            clean_msgs = []
            for m in messages:
                if m.get('role') == 'tool':
                    clean_msgs.append({
                        'role': 'user',
                        'content': (
                            f"Tool result ({m.get('name', 'unknown')}):\n"
                            f"{m['content']}"
                        )
                    })
                else:
                    clean_msgs.append(m)
            text = self.llm.tokenizer.apply_chat_template(
                clean_msgs, tokenize=False, add_generation_prompt=True
            )

        out = self.llm(
            text, max_new_tokens=max_new_tokens, temperature=0.1,
            do_sample=True, return_full_text=False,
        )
        return out[0]['generated_text'].strip()

    def _extract_thought(self, raw):
        """Extract reasoning text before the <tool_call> or Action: tag."""
        # Text before <tool_call>
        if '<tool_call>' in raw:
            before = raw.split('<tool_call>')[0].strip()
            if before:
                return re.sub(r'^Thought:\s*', '', before, flags=re.IGNORECASE)

        # Text before "Action:" (fallback for non-function-calling output)
        if 'Action:' in raw:
            before = raw.split('Action:')[0].strip()
            if before:
                return re.sub(r'^Thought:\s*', '', before, flags=re.IGNORECASE)

        # If entire output is reasoning (no tool call found by caller)
        thought_m = re.search(r'Thought:\s*(.+?)(?=\n|$)', raw, re.DOTALL)
        if thought_m:
            return thought_m.group(1).strip()

        return ''

    def _parse_tool_call(self, raw):
        """
        Parse LLM output for structured tool calls.
        Priority:
          1. Qwen2.5 <tool_call>{"name":..., "arguments":{...}}</tool_call>
          2. Bare JSON with "name" key
          3. Free-text 'Action: tool_name' fallback
          4. Direct JSON with MBTI fields → implicit predict
        """
        # Strategy 1: Qwen2.5 native <tool_call> format
        m = re.search(
            r'<tool_call>\s*(\{.*?\})\s*</tool_call>', raw, re.DOTALL
        )
        if m:
            try:
                call = json.loads(m.group(1))
                name = call.get('name', '')
                args = call.get('arguments', {})
                if isinstance(args, str):
                    try:
                        args = json.loads(args)
                    except (json.JSONDecodeError, ValueError):
                        args = {}
                return self._normalize_tool_name(name), args
            except (json.JSONDecodeError, ValueError):
                pass

        # Strategy 2: Bare JSON with "name" key (partial function call)
        for match in re.finditer(r'\{[^{}]*\}', raw):
            try:
                obj = json.loads(match.group())
                if 'name' in obj and isinstance(obj.get('name'), str):
                    name = obj.get('name', '')
                    args = obj.get('arguments', {})
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except (json.JSONDecodeError, ValueError):
                            args = {}
                    return self._normalize_tool_name(name), args
            except (json.JSONDecodeError, ValueError):
                continue

        # Strategy 3: Free-text "Action: tool_name" fallback
        action_m = re.search(r'Action:\s*(\S+)', raw, re.IGNORECASE)
        if action_m:
            name = action_m.group(1).strip().lower().rstrip('.,;:')
            return self._normalize_tool_name(name), {}

        # Strategy 4: Direct JSON with MBTI fields → implicit predict
        for match in re.finditer(r'\{[^{}]+\}', raw):
            try:
                obj = json.loads(match.group())
                if 'type' in obj or 'ie' in obj:
                    return 'predict', obj
            except (json.JSONDecodeError, ValueError):
                continue

        return None, None

    def _normalize_tool_name(self, name):
        """Map aliases to canonical tool names."""
        alias = {
            'retrieve_similar_posts': 'retrieve_similar',
            'similar': 'retrieve_similar',
            'retrieve': 'retrieve_similar',
            'knowledge': 'retrieve_knowledge',
            'analyze': 'analyze_text',
            'experience': 'recall_experience',
            'recall': 'recall_experience',
            'finish': 'predict',
        }
        name = name.lower().strip()
        return alias.get(name, name)

    # ─────────────────────────────────────────
    #  FALLBACK & PARSING
    # ─────────────────────────────────────────
    def _fallback_predict(self, posts_text, query_embedding):
        """Direct prediction when ReAct loop fails to produce a valid result."""
        similar = tool_retrieve_similar_posts(query_embedding)
        knowledge = tool_retrieve_knowledge(query_embedding)
        features = tool_analyze_text(posts_text)

        sim_str = '\n'.join(
            f"  [{s['type']}] (sim={s['similarity']:.2f}): {s['text'][:200]}"
            for s in similar[:5]
        )
        know_str = '\n'.join(
            f"  [{k['label']}]: {k['content'][:200]}" for k in knowledge[:4]
        )
        feat_str = json.dumps(features, indent=2)

        messages = [
            {'role': 'system', 'content': (
                'You are an MBTI classifier. Respond with ONLY JSON:\n'
                '{"type":"XXXX","ie":"I or E","ie_conf":0.0-1.0,'
                '"sn":"S or N","sn_conf":0.0-1.0,'
                '"tf":"T or F","tf_conf":0.0-1.0,'
                '"jp":"J or P","jp_conf":0.0-1.0}'
            )},
            {'role': 'user', 'content': (
                f'Similar posts:\n{sim_str}\n\n'
                f'Knowledge:\n{know_str}\n\n'
                f'Features:\n{feat_str}\n\n'
                f'Posts:\n{posts_text[:1500]}\n\n'
                'Classify. Output ONLY JSON:'
            )},
        ]

        raw = self._call_llm_with_tools(messages, max_new_tokens=150)
        result = self._parse_json(raw)
        if result:
            return self._validate(result)

        return {
            'type': 'INFP', 'ie': 'I', 'ie_conf': 0.5,
            'sn': 'N', 'sn_conf': 0.5, 'tf': 'F', 'tf_conf': 0.5,
            'jp': 'P', 'jp_conf': 0.5,
        }

    def _parse_json(self, raw):
        """Try to extract a JSON object or MBTI letters from raw text."""
        for candidate in [raw, raw.split('\n')[0]]:
            try:
                m = re.search(r'\{[^{}]+\}', candidate)
                if m:
                    obj = json.loads(m.group())
                    if 'type' in obj or 'ie' in obj:
                        return obj
            except (json.JSONDecodeError, ValueError):
                continue

        result = {}
        for dim, valid in [('ie', 'IE'), ('sn', 'SN'),
                           ('tf', 'TF'), ('jp', 'JP')]:
            for ch in raw.upper():
                if ch in valid:
                    result[dim] = ch
                    result[f'{dim}_conf'] = 0.5
                    break
            if dim not in result:
                defaults = {'ie': 'I', 'sn': 'N', 'tf': 'T', 'jp': 'P'}
                result[dim] = defaults[dim]
                result[f'{dim}_conf'] = 0.5

        if result:
            result['type'] = (result.get('ie', 'I') + result.get('sn', 'N')
                              + result.get('tf', 'T') + result.get('jp', 'P'))
            return result
        return None

    def _validate(self, obj):
        """Validate and normalize parsed result."""
        t = obj.get('type', '').upper()
        for dim, idx, pair in [('ie', 0, 'IE'), ('sn', 1, 'SN'),
                                ('tf', 2, 'TF'), ('jp', 3, 'JP')]:
            letter = obj.get(dim, '').upper()
            if letter not in pair:
                letter = (t[idx].upper()
                          if len(t) > idx and t[idx].upper() in pair
                          else pair[0])
            obj[dim] = letter

            conf = obj.get(f'{dim}_conf', 0.5)
            try:
                conf = max(0.0, min(1.0, float(conf)))
            except (ValueError, TypeError):
                conf = 0.5
            obj[f'{dim}_conf'] = conf

        obj['type'] = (obj['ie'] + obj['sn'] + obj['tf'] + obj['jp']).upper()
        return obj


# ── Instantiate Agent ──
agent = MBTIClassificationAgent(
    llm_pipeline=llm_pipe,
    embed_model=embed_model,
    confidence_threshold=CONFIG['CONFIDENCE_THRESHOLD'],
    max_retries=CONFIG['MAX_RETRIES'],
    memory_conf_threshold=CONFIG.get('MEMORY_CONF_THRESHOLD', 0.80),
    max_react_steps=CONFIG.get('MAX_REACT_STEPS', 10),
)

print('✅ Agent initialized (Function Calling + ReAct)')
print(f'   LLM:            {LLM_MODEL_NAME}')
print(f'   Tool Binding:   Qwen2.5 native <tool_call> format')
print(f'   Autonomy:       LLM selects tool + specifies arguments')
print(f'   Tools:          {", ".join(t["function"]["name"] for t in TOOL_DEFINITIONS)}')
print(f'   Loop:           max_steps={agent.max_react_steps}, '
      f'conf_threshold={agent.conf_threshold}, max_retries={agent.max_retries}')
print(f'   Memory:         scratchpad (short-term) + experience store '
      f'(long-term, threshold={agent.memory_conf_threshold})')
print(f'   RAG:            post_index ({post_index.ntotal} vecs) + '
      f'knowledge_index ({knowledge_index.ntotal} chunks)')

## 🔬 Cell 8b — DEMO: Single User Predictions with Full Agent Trace

Run the agent on a **small sample** of test users to visualize the **autonomous Function Calling loop** — where the LLM generates structured `<tool_call>` invocations with dynamic arguments (e.g., targeted knowledge queries).

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  DEMO: Function Calling Agent Trace
# ═══════════════════════════════════════════════════════════════

DEMO_INDICES = [0, 50, 100]  # Pick 3 diverse test samples
NUM_DEMOS = len(DEMO_INDICES)

DIM_LETTER_MAP_INV = {
    'ie': {1: 'I', 0: 'E'},
    'sn': {1: 'N', 0: 'S'},
    'tf': {1: 'F', 0: 'T'},
    'jp': {1: 'J', 0: 'P'},
}

# Reset all agent memory — watch it grow across demos
agent.pattern_memory = {}
agent.experience_store = []
agent.experience_index = None
agent.total_memories_stored = 0
agent.total_retries = 0

print('=' * 70)
print(f'🔬 DEMO: {NUM_DEMOS} Users — Function Calling Agent Trace')
print('=' * 70)
print(f'   LLM generates <tool_call> with name + arguments autonomously.')
print(f'   retrieve_knowledge uses LLM-specified queries (dynamic embedding).')
print(f'   Long-term memory starts EMPTY — watch it grow!\n')

for demo_num, sample_idx in enumerate(DEMO_INDICES, 1):
    if sample_idx >= len(test_df):
        print(f'\n⚠️  Sample index {sample_idx} out of range, skipping.')
        continue

    row = test_df.iloc[sample_idx]
    posts_text = row['concat_posts']
    query_emb  = test_embeddings[sample_idx]

    # Ground truth
    true_type = row[LABEL_COL].upper()
    true_dims = {
        'ie': DIM_LETTER_MAP_INV['ie'][row['label_ie']],
        'sn': DIM_LETTER_MAP_INV['sn'][row['label_sn']],
        'tf': DIM_LETTER_MAP_INV['tf'][row['label_tf']],
        'jp': DIM_LETTER_MAP_INV['jp'][row['label_jp']],
    }

    # Run agent (memory accumulates between demos!)
    t0 = time.time()
    result = agent.classify(posts_text, query_emb)
    elapsed = time.time() - t0

    pred_type = result.get('type', '????')

    # ── Header ──
    print(f'\n{"=" * 70}')
    print(f'  🧑 DEMO {demo_num}/{NUM_DEMOS}  |  Test Sample #{sample_idx}')
    print(f'{"=" * 70}')

    # ── Posts preview ──
    preview = posts_text[:300].replace('\n', ' ')
    print(f'\n📝 Posts preview:\n   "{preview}..."')

    # ── True vs Predicted ──
    match_icon = '✅' if pred_type == true_type else '❌'
    print(f'\n📋 True MBTI:      {true_type}')
    print(f'🤖 Predicted MBTI: {pred_type}  {match_icon}')
    print(f'⏱  Inference time: {elapsed:.1f}s')

    # ── Per-axis detail ──
    print(f'\n📊 Axis Predictions:')
    dim_full = {'ie': 'I/E', 'sn': 'S/N', 'tf': 'T/F', 'jp': 'J/P'}
    for key in ['ie', 'sn', 'tf', 'jp']:
        pred_letter = result.get(key, '?')
        conf = result.get(f'{key}_conf', 0.0)
        true_letter = true_dims[key]
        axis_match = '✅' if pred_letter == true_letter else '❌'
        conf_bar = '█' * int(conf * 20) + '░' * (20 - int(conf * 20))
        print(f'   {dim_full[key]}: {pred_letter} (confidence: {conf:.2f}) '
              f'[{conf_bar}] true={true_letter} {axis_match}')

    # ── Agent Trace with Function Call details ──
    react_steps = [s for s in agent.trace if s.get('llm_generated')]
    sys_steps   = [s for s in agent.trace if not s.get('llm_generated')]
    print(f'\n🔄 Agent Trace ({len(react_steps)} LLM function calls, '
          f'{len(sys_steps)} system steps):')

    for i, step in enumerate(agent.trace):
        step_name = step.get('step', 'unknown')
        thought   = step.get('thought', '')
        tool_args = step.get('tool_args', {})
        obs       = step.get('observation', '')
        is_llm    = step.get('llm_generated', False)

        icons = {
            'analyze_text': '🔬', 'retrieve_similar': '🔍',
            'retrieve_knowledge': '📚', 'recall_experience': '💡',
            'predict': '🧠', 'memory_update': '💾',
            'fallback_predict': '⚠️',
        }
        icon = icons.get(step_name, '▶')

        source = '🤖 LLM Function Call' if is_llm else '⚙️ System'
        print(f'\n   Step {i+1}: {icon} [{step_name}] ({source})')

        # Show LLM reasoning (text before <tool_call>)
        if thought:
            display_thought = thought[:400] + '...' if len(thought) > 400 else thought
            print(f'   💭 Thought: {display_thought}')

        # Show function call arguments (key autonomy feature!)
        if tool_args and is_llm:
            args_str = json.dumps(tool_args, ensure_ascii=False)
            if len(args_str) > 200:
                args_str = args_str[:200] + '...'
            print(f'   🔧 Args: {args_str}')

        # Show observation (truncated)
        if obs:
            display_obs = obs[:300] + '...' if len(obs) > 300 else obs
            print(f'   👁  Observation: {display_obs}')

    # ── Memory summary ──
    retries = agent.scratchpad.get('retries', 0)
    if retries > 0:
        print(f'\n   ⚡ Agent re-predicted {retries} time(s) due to low confidence')
    else:
        print(f'\n   ⚡ No re-predictions needed — confident on first predict')

    print(f'\n   🧠 Long-term memory: {len(agent.experience_store)} experiences, '
          f'types: {dict(agent.pattern_memory)}')
    print(f'{"─" * 70}')

# ── Final memory summary ──
print(f'\n{"=" * 70}')
print(f'📊 Long-term Memory Summary after {NUM_DEMOS} Demos:')
print(f'   Experiences stored:  {len(agent.experience_store)}/{NUM_DEMOS}')
print(f'   Type distribution:   '
      f'{dict(sorted(agent.pattern_memory.items(), key=lambda x: -x[1]))}')
if agent.experience_store:
    print(f'   Stored experiences:')
    for i, exp in enumerate(agent.experience_store):
        print(f'     #{i+1}: {exp["type"]} (avg_conf={exp["avg_conf"]:.2f})')
        print(f'          LLM Reasoning: {exp["reasoning"][:200]}...')
print(f'{"=" * 70}')

# Reset memory for full test run
agent.pattern_memory = {}
agent.experience_store = []
agent.experience_index = None
agent.total_memories_stored = 0
agent.total_retries = 0
print(f'\n🔬 Demo complete. Agent memory reset for full test run.')

## ⚡ Cell 8c — Batched Inference Engine

**Problem:** Sequential inference = ~103s/sample × 694 = ~20 hours.

**Solution:** Batch N samples together → 1 `model.generate()` call processes all N simultaneously.

| Approach | LLM calls per step | Time per step | Total (694 samples) |
|----------|-------------------|---------------|---------------------|
| Sequential (batch=1) | 1 | ~20s | ~20h |
| **Batched (batch=4)** | **4 in parallel** | **~25s** | **~5-6h** |
| Batched (batch=8) | 8 in parallel | ~30s | **~3-4h** |

Key techniques:
- **Left-padding** for correct batched autoregressive generation
- **OOM fallback** — if batch OOMs, automatically falls back to sequential
- **Parallel ReAct states** — each sample has its own conversation history, all batched at each step

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  BATCHED INFERENCE ENGINE
#  Process N samples in parallel via batched model.generate()
#  Speedup: ~3-4x by batching LLM calls across samples
# ═══════════════════════════════════════════════════════════════

def batch_generate(pipe, text_prompts, max_new_tokens=200):
    """
    Batched LLM generation: run N prompts in a single model.generate() call.
    Uses left-padding for correct batched autoregressive generation.
    Falls back to sequential pipeline calls if GPU OOMs.
    """
    if len(text_prompts) == 1:
        out = pipe(
            text_prompts[0], max_new_tokens=max_new_tokens,
            temperature=0.1, do_sample=True, return_full_text=False,
        )
        return [out[0]['generated_text'].strip()]

    tokenizer = pipe.tokenizer
    model = pipe.model

    # Left-padding is REQUIRED for correct batched generation
    orig_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # Tokenize batch with padding
    encoded = tokenizer(
        text_prompts, return_tensors='pt', padding=True,
        truncation=True, max_length=3072
    )
    input_length = encoded['input_ids'].shape[1]

    # Move to model device (handles device_map='auto')
    try:
        device = model.device
        if hasattr(device, 'type') and device.type == 'meta':
            device = 'cuda:0'
    except Exception:
        device = 'cuda:0'
    encoded = {k: v.to(device) for k, v in encoded.items()}

    try:
        with torch.no_grad():
            output_ids = model.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )

        # Decode only the newly generated tokens
        results = []
        for i in range(len(text_prompts)):
            new_ids = output_ids[i][input_length:]
            text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
            results.append(text)

    except torch.cuda.OutOfMemoryError:
        # Fallback: sequential processing if batch OOMs
        torch.cuda.empty_cache()
        results = []
        for prompt in text_prompts:
            out = pipe(
                prompt, max_new_tokens=max_new_tokens,
                temperature=0.1, do_sample=True, return_full_text=False,
            )
            results.append(out[0]['generated_text'].strip())

    tokenizer.padding_side = orig_padding_side
    torch.cuda.empty_cache()
    return results


def classify_batch(agent, batch_posts, batch_embeddings, max_new_tokens=200):
    """
    Process N samples simultaneously with batched LLM generation.

    Instead of sequential:
      sample1_step1 → sample1_step2 → ... → sample2_step1 → ...
    Batched:
      [sample1_step1, sample2_step1, ...] → [sample1_step2, sample2_step2, ...] → ...

    At each ReAct step, ALL active samples' prompts are batched into
    a single model.generate() call. Tool execution (FAISS, text analysis)
    is sequential but negligible (<1ms each).
    """
    n = len(batch_posts)

    # ── Initialize parallel agent states ──
    states = []
    for i in range(n):
        states.append({
            'posts_text': batch_posts[i],
            'query_embedding': batch_embeddings[i],
            'messages': [
                {'role': 'system', 'content': AGENT_SYSTEM_PROMPT},
                {'role': 'user', 'content': (
                    "Classify this user's MBTI type based on their forum posts:\n\n"
                    + batch_posts[i][:2000]
                )},
            ],
            'result': None,
            'done': False,
            'tools_called': set(),
            'predict_attempts': 0,
            'trace': [],
        })

    # ── Batched ReAct Loop ──
    for step in range(agent.max_react_steps):
        active = [s for s in states if not s['done']]
        if not active:
            break

        # Build formatted prompts for all active samples
        prompts = []
        for s in active:
            try:
                text = agent.llm.tokenizer.apply_chat_template(
                    s['messages'], tools=TOOL_DEFINITIONS,
                    tokenize=False, add_generation_prompt=True
                )
            except TypeError:
                clean = []
                for m in s['messages']:
                    if m.get('role') == 'tool':
                        clean.append({
                            'role': 'user',
                            'content': f"Tool result ({m.get('name','')}):\n{m['content']}"
                        })
                    else:
                        clean.append(m)
                text = agent.llm.tokenizer.apply_chat_template(
                    clean, tokenize=False, add_generation_prompt=True
                )
            prompts.append(text)

        # ═══ BATCHED LLM CALL (core speedup) ═══
        raws = batch_generate(agent.llm, prompts, max_new_tokens=max_new_tokens)

        # Process each result
        for s, raw in zip(active, raws):
            thought = agent._extract_thought(raw)
            tool_name, tool_args = agent._parse_tool_call(raw)

            if not tool_name:
                if 'analyze_text' not in s['tools_called']:
                    tool_name, tool_args = 'analyze_text', {}
                elif 'retrieve_similar' not in s['tools_called']:
                    tool_name, tool_args = 'retrieve_similar', {}
                else:
                    tool_name, tool_args = 'predict', {}

            s['trace'].append({
                'step': tool_name,
                'thought': thought or '(direct tool call)',
                'tool_args': tool_args or {},
                'llm_generated': True,
            })

            if tool_name == 'predict':
                result = None
                if tool_args and ('type' in tool_args or 'ie' in tool_args):
                    result = agent._validate(tool_args)
                else:
                    result = agent._parse_json(raw)
                    if result:
                        result = agent._validate(result)

                if result:
                    s['predict_attempts'] += 1
                    low_conf = [
                        d for d in ['ie', 'sn', 'tf', 'jp']
                        if result.get(f'{d}_conf', 0) < agent.conf_threshold
                    ]
                    if not low_conf or s['predict_attempts'] > agent.max_retries:
                        s['result'] = result
                        s['done'] = True
                    else:
                        dim_names = {'ie': 'I/E', 'sn': 'S/N', 'tf': 'T/F', 'jp': 'J/P'}
                        low_str = ', '.join(
                            f"{dim_names[d]}={result[f'{d}_conf']:.2f}"
                            for d in low_conf
                        )
                        obs = (
                            f'Prediction {result["type"]} has LOW CONFIDENCE on '
                            f'{low_str}. Call retrieve_knowledge with targeted '
                            f'query about uncertain dimension(s), then re-predict.'
                        )
                        s['messages'].append({'role': 'assistant', 'content': raw})
                        s['messages'].append({'role': 'user', 'content': f'Observation: {obs}'})
                        agent.total_retries += 1
                else:
                    obs = 'Could not parse prediction. Call predict with valid arguments.'
                    s['messages'].append({'role': 'assistant', 'content': raw})
                    s['messages'].append({'role': 'user', 'content': f'Observation: {obs}'})
            else:
                s['tools_called'].add(tool_name)
                observation = agent._execute_tool(
                    tool_name, tool_args, s['posts_text'], s['query_embedding']
                )
                if observation is None:
                    obs_str = (
                        f'Unknown tool "{tool_name}". Available: '
                        f'analyze_text, retrieve_similar, '
                        f'retrieve_knowledge, recall_experience, predict'
                    )
                else:
                    obs_str = agent._format_observation(tool_name, observation)

                s['messages'].append({'role': 'assistant', 'content': raw})
                s['messages'].append({
                    'role': 'tool', 'name': tool_name, 'content': obs_str
                })

    # ── Finalize: fallbacks + memory updates ──
    results = []
    for s in states:
        if s['result'] is None:
            s['result'] = agent._fallback_predict(
                s['posts_text'], s['query_embedding']
            )
            s['trace'].append({
                'step': 'fallback_predict',
                'thought': 'Loop exhausted. Fallback prediction.',
                'tool_args': {},
                'llm_generated': False,
            })

        # Memory update
        mbti = s['result'].get('type', 'INFP')
        agent.pattern_memory[mbti] = agent.pattern_memory.get(mbti, 0) + 1
        agent.trace = s['trace']
        agent._store_experience(
            s['query_embedding'], s['posts_text'], s['result']
        )

        results.append(s['result'])

    return results


BATCH_SIZE = CONFIG.get('BATCH_SIZE', 4)
print(f'✅ Batched inference engine ready (batch_size={BATCH_SIZE})')
print(f'   batch_generate:  N prompts → 1 model.generate() call')
print(f'   classify_batch:  parallel ReAct states + batched LLM calls')
print(f'   Expected speedup: ~{BATCH_SIZE-1}-{BATCH_SIZE}x vs sequential')

## 🚀 Cell 9 — Run Agent Inference on Test Set (Batched)

Processes `BATCH_SIZE` samples in parallel at each ReAct step via batched `model.generate()`. Adjust `BATCH_SIZE` in Config (default=4, increase to 8 if GPU memory allows).

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  RUN AGENT INFERENCE ON TEST SET (BATCHED)
# ═══════════════════════════════════════════════════════════════
n_test = len(test_df)
n_batches = (n_test + BATCH_SIZE - 1) // BATCH_SIZE

# Estimate: sequential ~103s/sample, batched ~103/BATCH_SIZE s/sample
est_seq_h = n_test * 103 / 3600
est_batch_h = est_seq_h / max(BATCH_SIZE - 1, 1)

print(f'⏳ [RAG+Agent] Batched inference on {n_test} test samples...')
print(f'   Batch size:      {BATCH_SIZE} (parallel LLM generation)')
print(f'   Total batches:   {n_batches}')
print(f'   ReAct loop:      ~3-5 LLM calls per sample (autonomous tool selection)')
print(f'   Sequential est:  ~{est_seq_h:.1f}h')
print(f'   Batched est:     ~{est_batch_h:.1f}h (≈{BATCH_SIZE-1}-{BATCH_SIZE}x speedup)\n')

agent_results = []
errors = 0
t_start = time.time()

for batch_idx in tqdm(range(n_batches), desc='Batch inference'):
    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, n_test)

    batch_posts = [test_df.iloc[i]['concat_posts'] for i in range(start, end)]
    batch_embs = test_embeddings[start:end]

    try:
        batch_results = classify_batch(agent, batch_posts, batch_embs)
        agent_results.extend(batch_results)
    except Exception as e:
        # On error, fill with defaults and continue
        errors += (end - start)
        for _ in range(end - start):
            agent_results.append({
                'type': 'INFP', 'ie': 'I', 'ie_conf': 0.5,
                'sn': 'N', 'sn_conf': 0.5, 'tf': 'F', 'tf_conf': 0.5,
                'jp': 'P', 'jp_conf': 0.5,
            })
        if errors <= 8:  # Print first few errors for debugging
            print(f'  ⚠️ Batch {batch_idx} error: {str(e)[:200]}')

    # Progress report every 25 batches (~100 samples)
    if (batch_idx + 1) % 25 == 0:
        elapsed = time.time() - t_start
        done = len(agent_results)
        speed = done / elapsed
        eta = (n_test - done) / speed if speed > 0 else 0
        top_types = sorted(agent.pattern_memory.items(), key=lambda x: -x[1])[:5]
        print(f'  [{done}/{n_test}] '
              f'speed={speed:.2f} samples/s | ETA={eta/60:.1f}min | '
              f'errors={errors} | retries={agent.total_retries}')
        print(f'    Top types: {top_types}')

elapsed_total = time.time() - t_start
print(f'\n✅ Inference complete in {elapsed_total/60:.1f} minutes')
print(f'   Speed: {n_test/elapsed_total:.2f} samples/s '
      f'(batch_size={BATCH_SIZE}, '
      f'{BATCH_SIZE * n_test/elapsed_total:.1f}x vs est. sequential)')
print(f'   Errors: {errors}/{n_test}')
print(f'   Agent retries: {agent.total_retries}')
print(f'   Experiences stored: {len(agent.experience_store)} '
      f'({len(agent.experience_store)/n_test*100:.1f}%)')
print(f'   Type distribution:')
for t, cnt in sorted(agent.pattern_memory.items(), key=lambda x: -x[1]):
    print(f'     {t}: {cnt} ({cnt/n_test*100:.1f}%)')

## 📊 Cell 10 — Metrics (F1, Accuracy, AUC) + Save CSV

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  METRICS: F1, Accuracy, AUC  —  4 axes + full 16 types
# ═══════════════════════════════════════════════════════════════

# ── Extract predictions into arrays ──
DIM_LABEL_MAP = {
    'ie': {'I': 1, 'E': 0},
    'sn': {'N': 1, 'S': 0},
    'tf': {'F': 1, 'T': 0},
    'jp': {'J': 1, 'P': 0},
}
DIM_KEYS = ['ie', 'sn', 'tf', 'jp']

y_true_axes = test_df[LABEL_COLS].values           # [N, 4]
y_pred_axes = np.zeros((len(agent_results), 4), dtype=int)
y_prob_axes = np.zeros((len(agent_results), 4), dtype=float)

for i, res in enumerate(agent_results):
    for j, key in enumerate(DIM_KEYS):
        letter = res.get(key, 'I' if j == 0 else 'N' if j == 1 else 'F' if j == 2 else 'P')
        pred   = DIM_LABEL_MAP[key].get(letter, 0)
        conf   = res.get(f'{key}_conf', 0.5)
        y_pred_axes[i, j] = pred
        # P(class=1): if predicted class=1 → prob=conf; if predicted class=0 → prob=1-conf
        y_prob_axes[i, j] = conf if pred == 1 else (1.0 - conf)


# ═══════════════ 4-AXIS METRICS ═══════════════
print('=' * 70)
print('📊  4-AXIS METRICS — RAG + LLM Agent')
print('=' * 70)
print(f'{"Axis":<8} {"F1-Macro":<12} {"Accuracy":<12} {"AUC-ROC":<12}')
print('-' * 44)

f1_list, acc_list, auc_list = [], [], []
for i, name in enumerate(DIM_NAMES):
    f1  = f1_score(y_true_axes[:, i], y_pred_axes[:, i], average='macro', zero_division=0)
    acc = accuracy_score(y_true_axes[:, i], y_pred_axes[:, i])
    try:
        auc = roc_auc_score(y_true_axes[:, i], y_prob_axes[:, i])
    except ValueError:
        auc = 0.5
    f1_list.append(f1); acc_list.append(acc); auc_list.append(auc)
    print(f'{name:<8} {f1:<12.4f} {acc:<12.4f} {auc:<12.4f}')

print('-' * 44)
print(f'{"Average":<8} {np.mean(f1_list):<12.4f} {np.mean(acc_list):<12.4f} '
      f'{np.mean(auc_list):<12.4f}')


# ═══════════════ 16-TYPE METRICS ═══════════════
y_true_type = test_df[LABEL_COL].str.upper().tolist()
y_pred_type = [r['type'].upper() for r in agent_results]

type_to_idx = {t.upper(): i for i, t in enumerate(MBTI_TYPES)}
y_true_16 = np.array([type_to_idx.get(t, 0) for t in y_true_type])
y_pred_16 = np.array([type_to_idx.get(t, 0) for t in y_pred_type])

# Approximate 16-class probabilities from 4-axis confidences
# P(TYPE) ≈ P(dim1_match) × P(dim2_match) × P(dim3_match) × P(dim4_match)
y_prob_16 = np.zeros((len(agent_results), 16), dtype=float)
for i in range(len(agent_results)):
    for t_idx, t_name in enumerate(MBTI_TYPES):
        t_upper = t_name.upper()
        p = 1.0
        for j, (key, letter) in enumerate(zip(DIM_KEYS, t_upper)):
            target = DIM_LABEL_MAP[key].get(letter, 0)
            p *= y_prob_axes[i, j] if target == 1 else (1.0 - y_prob_axes[i, j])
        y_prob_16[i, t_idx] = p
    row_sum = y_prob_16[i].sum()
    if row_sum > 0:
        y_prob_16[i] /= row_sum

f1_16_macro    = f1_score(y_true_16, y_pred_16, average='macro', zero_division=0)
f1_16_weighted = f1_score(y_true_16, y_pred_16, average='weighted', zero_division=0)
acc_16         = accuracy_score(y_true_16, y_pred_16)

try:
    y_true_16_bin = label_binarize(y_true_16, classes=list(range(16)))
    auc_16 = roc_auc_score(y_true_16_bin, y_prob_16, average='macro', multi_class='ovr')
except Exception:
    auc_16 = 0.0

print(f'\n{"=" * 70}')
print('📊  16-TYPE METRICS — RAG + LLM Agent')
print(f'{"=" * 70}')
print(f'  Accuracy:            {acc_16:.4f}')
print(f'  F1 (Macro):          {f1_16_macro:.4f}')
print(f'  F1 (Weighted):       {f1_16_weighted:.4f}')
print(f'  AUC-ROC (Macro OvR): {auc_16:.4f}')

# Per-type breakdown
present_labels = sorted(set(y_true_16.tolist()) | set(y_pred_16.tolist()))
target_names   = [MBTI_TYPES[i].upper() for i in present_labels]
print(f'\n📋 Per-Type Classification Report:')
print(classification_report(
    y_true_16, y_pred_16,
    labels=present_labels,
    target_names=target_names,
    zero_division=0
))


# ═══════════════ SAVE TO CSV ═══════════════
agent_save = pd.DataFrame()
for i, c in enumerate(LABEL_COLS):
    agent_save[f'y_true_{c}'] = y_true_axes[:, i]
    agent_save[f'y_pred_{c}'] = y_pred_axes[:, i]
    agent_save[f'y_prob_{c}'] = np.round(y_prob_axes[:, i], 4)

agent_save['y_true_type'] = y_true_type
agent_save['y_pred_type'] = y_pred_type

agent_save.to_csv(f"{CONFIG['RESULT_DIR']}/rag_agent_predictions.csv", index=False)
print(f'\n✅ Saved → {CONFIG["RESULT_DIR"]}/rag_agent_predictions.csv')